# Swasthya — Feature 1: Pharmacy Demand Forecasting

**What this notebook does:**  
Trains a LightGBM model on 18 months of pharmacy sales data to predict how much of each product a pharmacy will sell in the next 7 days — then generates a reorder suggestion with a Gemini-written explanation.

**Output of this notebook:**
- `demand_forecaster.pkl` — trained LightGBM model
- `feature_scaler.pkl` — StandardScaler for features  
- `model_metadata.json` — metrics + feature list for FastAPI
- `reorder_suggestions.csv` — demo output for all pharmacies right now

---
```
STEP 1  → Install & import
STEP 2  → Load your CSV data
STEP 3  → Exploratory analysis
STEP 4  → Feature engineering
STEP 5  → Train LightGBM model
STEP 6  → Evaluate (MAPE, RMSE, R²)
STEP 7  → Compare vs baseline (Holt-Winters)
STEP 8  → Predict next 7 days demand
STEP 9  → Generate reorder suggestions
STEP 10 → Gemini explainability
STEP 11 → Save all model files
STEP 12 → FastAPI-ready inference function
```

## Install & Import

In [ ]:
# Install required packages
!pip install lightgbm scikit-learn statsmodels google-generativeai --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import joblib
import json
import warnings
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

os.makedirs('/content/ML/models', exist_ok=True)

print('✅ All packages imported')

## Load CSV Data

In [ ]:
# ── Option A: Upload the zip file from your computer ─────────────────────────
# Run this block if you downloaded swasthya_dataset.zip

from google.colab import files
import zipfile

print('📂 Upload swasthya_dataset.zip when the dialog appears...')
uploaded = files.upload()

with zipfile.ZipFile('swasthya_dataset.zip', 'r') as z:
    z.extractall('/content/')

print('✅ Files extracted')

In [ ]:
# ── Load all CSVs ─────────────────────────────────────────────────────────────
sales_df = pd.read_csv(
    '/content/swasthya_data/daily_sales.csv',
    parse_dates=['date']
)

inventory_df = pd.read_csv('/content/swasthya_data/pharmacy_inventory.csv')
products_df  = pd.read_csv('/content/swasthya_data/master_products.csv')
pharmacy_df  = pd.read_csv('/content/swasthya_data/master_pharmacies.csv')

# ── Quick sanity check ────────────────────────────────────────────────────────
print('='*55)
print('  SWASTHYA — FEATURE 1 DATA LOADED')
print('='*55)
print(f'  Sales rows   : {len(sales_df):,}')
print(f'  Date range   : {sales_df["date"].min().date()} → {sales_df["date"].max().date()}')
print(f'  Pharmacies   : {sales_df["pharmacy_id"].nunique()}')
print(f'  SKUs         : {sales_df["sku_id"].nunique()}')
print(f'  Products     : {sales_df["product_name"].unique().tolist()}')
print(f'  Total revenue: ₹{sales_df["revenue"].sum()/1e6:.1f}M')
print('='*55)
print()
sales_df.head(3)

## Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🛍️  Swasthya — Sales EDA (Reusable Hygiene Products)', fontsize=15, fontweight='bold')

# 1. Monthly total sales trend
monthly = (
    sales_df
    .groupby(sales_df['date'].dt.to_period('M'))['quantity_sold']
    .sum()
    .reset_index()
)
monthly['date'] = monthly['date'].dt.to_timestamp()
axes[0,0].plot(monthly['date'], monthly['quantity_sold'], color='#7B1FA2', linewidth=2.5)
axes[0,0].fill_between(monthly['date'], monthly['quantity_sold'], alpha=0.15, color='#7B1FA2')
axes[0,0].set_title('Monthly Sales Volume — All Products')
axes[0,0].set_ylabel('Units Sold')
axes[0,0].tick_params(axis='x', rotation=30)
# Annotate MHD spike
axes[0,0].axvspan(
    pd.Timestamp('2024-05-21'), pd.Timestamp('2024-06-07'),
    alpha=0.2, color='red', label='MHD Window'
)
axes[0,0].legend(fontsize=8)

# 2. Revenue by product
prod_rev = (
    sales_df
    .groupby('product_name')['revenue']
    .sum()
    .sort_values(ascending=True)
)
short_names = [n.replace(' with ', '\nw/ ').replace(' Pack', '\nPack') for n in prod_rev.index]
colors = plt.cm.RdPu(np.linspace(0.3, 0.9, len(prod_rev)))
axes[0,1].barh(range(len(prod_rev)), prod_rev.values / 1e6, color=colors)
axes[0,1].set_yticks(range(len(prod_rev)))
axes[0,1].set_yticklabels(short_names, fontsize=8)
axes[0,1].set_title('Total Revenue by Product')
axes[0,1].set_xlabel('Revenue (₹ Millions)')

# 3. Day-of-week demand pattern
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow = sales_df.groupby('day_of_week')['quantity_sold'].mean()
bar_colors = ['#AB47BC' if i < 5 else '#CE93D8' for i in range(7)]
axes[0,2].bar(dow_labels, dow.values, color=bar_colors)
axes[0,2].set_title('Avg Daily Demand — Day of Week')
axes[0,2].set_ylabel('Avg Units Sold')

# 4. Product sales over time (each product)
for prod_id in sales_df['sku_id'].unique():
    prod_monthly = (
        sales_df[sales_df['sku_id']==prod_id]
        .groupby(sales_df['date'].dt.to_period('M'))['quantity_sold']
        .sum()
        .reset_index()
    )
    prod_monthly['date'] = prod_monthly['date'].dt.to_timestamp()
    prod_name = sales_df[sales_df['sku_id']==prod_id]['product_name'].iloc[0]
    short = prod_name.split(' ')[0] + ' ' + prod_name.split(' ')[1]
    axes[1,0].plot(prod_monthly['date'], prod_monthly['quantity_sold'], label=short, linewidth=1.5)
axes[1,0].set_title('Monthly Sales — Per Product')
axes[1,0].set_ylabel('Units Sold')
axes[1,0].legend(fontsize=6.5, ncol=2)
axes[1,0].tick_params(axis='x', rotation=30)

# 5. City tier comparison
tier_rev = sales_df.groupby('city_tier')['revenue'].sum()
tier_labels = [f'Tier {t}' for t in tier_rev.index]
axes[1,1].bar(tier_labels, tier_rev.values/1e6, color=['#6A1B9A','#AB47BC','#CE93D8'])
axes[1,1].set_title('Revenue by City Tier')
axes[1,1].set_ylabel('Revenue (₹ Millions)')

# 6. Seasonal events impact
sales_df['event'] = 'Normal'
sales_df.loc[sales_df['is_mhd_window']==1, 'event'] = 'MHD Window'
sales_df.loc[sales_df['is_ngo_season']==1, 'event'] = 'NGO Season'
sales_df.loc[sales_df['is_monsoon']==1, 'event'] = 'Monsoon'
event_avg = sales_df.groupby('event')['quantity_sold'].mean().sort_values(ascending=False)
ev_colors = {'MHD Window':'#C62828','NGO Season':'#1565C0','Normal':'#7B1FA2','Monsoon':'#558B2F'}
axes[1,2].bar(event_avg.index, event_avg.values,
              color=[ev_colors.get(e,'gray') for e in event_avg.index])
axes[1,2].set_title('Avg Daily Demand by Season/Event')
axes[1,2].set_ylabel('Avg Units/Day')

plt.tight_layout()
plt.savefig('/content/ML/models/eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA complete — saved eda_charts.png')

## Feature Engineering

In [ ]:
def engineer_features(df):
    """
    Takes raw daily sales dataframe.
    Returns dataframe with 28 engineered features ready for LightGBM.
    """
    df = df.sort_values(['pharmacy_id', 'sku_id', 'date']).copy()
    grp = df.groupby(['pharmacy_id', 'sku_id'])['quantity_sold']

    # ── LAG FEATURES ─────────────────────────────────────────────────────────
    # How much sold N days ago — the model learns "last week was high so this
    # week might also be high"
    df['lag_1']   = grp.shift(1)    # yesterday
    df['lag_7']   = grp.shift(7)    # same weekday last week
    df['lag_14']  = grp.shift(14)   # 2 weeks ago
    df['lag_28']  = grp.shift(28)   # 4 weeks ago
    df['lag_91']  = grp.shift(91)   # ~3 months ago (catches quarterly NGO patterns)
    df['lag_365'] = grp.shift(365)  # same day last year (annual seasonality)

    # ── ROLLING STATISTICS ────────────────────────────────────────────────────
    # Recent average demand — smooth out daily noise
    df['roll_7_mean']   = grp.transform(lambda x: x.shift(1).rolling(7,  min_periods=3).mean())
    df['roll_7_std']    = grp.transform(lambda x: x.shift(1).rolling(7,  min_periods=3).std())
    df['roll_14_mean']  = grp.transform(lambda x: x.shift(1).rolling(14, min_periods=7).mean())
    df['roll_30_mean']  = grp.transform(lambda x: x.shift(1).rolling(30, min_periods=14).mean())
    df['roll_60_mean']  = grp.transform(lambda x: x.shift(1).rolling(60, min_periods=30).mean())
    df['roll_90_mean']  = grp.transform(lambda x: x.shift(1).rolling(90, min_periods=45).mean())

    # ── TREND FEATURES ────────────────────────────────────────────────────────
    # Is demand accelerating or slowing down?
    df['trend_7_vs_30']   = df['roll_7_mean']  / (df['roll_30_mean']  + 1e-6)
    df['trend_14_vs_60']  = df['roll_14_mean'] / (df['roll_60_mean']  + 1e-6)
    df['trend_30_vs_90']  = df['roll_30_mean'] / (df['roll_90_mean']  + 1e-6)

    # ── CYCLICAL TIME ENCODING ────────────────────────────────────────────────
    # sin/cos tells the model that Dec 31 and Jan 1 are CLOSE — a raw number
    # 12 and 1 would not convey that
    df['sin_dow']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['cos_dow']  = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['sin_doy']  = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['cos_doy']  = np.cos(2 * np.pi * df['day_of_year'] / 365)
    df['sin_month']= np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month']= np.cos(2 * np.pi * df['month'] / 12)

    # ── ENTITY ENCODING ───────────────────────────────────────────────────────
    df['sku_code']  = df['sku_id'].str.extract(r'(\d+)').astype(int)
    df['ph_code']   = df['pharmacy_id'].str.extract(r'(\d+)').astype(int)
    df['city_code'] = df['city'].astype('category').cat.codes
    df['cat_code']  = df['category'].astype('category').cat.codes

    return df.dropna(subset=[
        'lag_1','lag_7','roll_7_mean','roll_30_mean'
    ])


print('⏳ Engineering features...')
feat_df = engineer_features(sales_df)

# ── Define feature columns used by model ─────────────────────────────────────
FEATURE_COLS = [
    # Lags
    'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_91',
    # Rolling
    'roll_7_mean', 'roll_7_std', 'roll_14_mean',
    'roll_30_mean', 'roll_60_mean', 'roll_90_mean',
    # Trends
    'trend_7_vs_30', 'trend_14_vs_60', 'trend_30_vs_90',
    # Cyclical time
    'sin_dow', 'cos_dow', 'sin_doy', 'cos_doy',
    'sin_month', 'cos_month',
    # Calendar
    'month', 'quarter', 'city_tier',
    'is_weekend', 'is_mhd_window', 'is_ngo_season', 'is_monsoon',
    # Entity
    'sku_code', 'ph_code', 'city_code', 'cat_code',
]
TARGET_COL = 'quantity_sold'

print(f'✅ Feature engineering complete')
print(f'   Original rows : {len(sales_df):,}')
print(f'   After dropna  : {len(feat_df):,}')
print(f'   Features used : {len(FEATURE_COLS)}')
feat_df[FEATURE_COLS].describe().round(3)

## Train LightGBM Model

In [ ]:
# ── Time-based train / test split ────────────────────────────────────────────
# IMPORTANT: Always split on time, never randomly.
# Random split would let the model "see the future" during training.
# We use last 60 days as test set.

cutoff = feat_df['date'].max() - pd.Timedelta(days=60)
train_df = feat_df[feat_df['date'] <= cutoff]
test_df  = feat_df[feat_df['date'] >  cutoff]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print(f'Train: {len(X_train):,} rows | {train_df["date"].min().date()} → {train_df["date"].max().date()}')
print(f'Test : {len(X_test):,} rows  | {test_df["date"].min().date()} → {test_df["date"].max().date()}')
print()

# ── LightGBM Model ────────────────────────────────────────────────────────────
# Why LightGBM?
# - Handles mixed feature types well (integers, floats, binary flags)
# - Faster than XGBoost on tabular data
# - Built-in early stopping avoids overfitting
# - Feature importance is easy to extract for explainability

model = lgb.LGBMRegressor(
    n_estimators      = 1000,   # max trees — early stopping will reduce this
    learning_rate     = 0.03,   # slow learning = better generalization
    num_leaves        = 63,     # tree complexity
    max_depth         = 7,
    min_child_samples = 20,     # prevents overfitting on small groups
    subsample         = 0.8,    # row sampling per tree
    colsample_bytree  = 0.8,    # feature sampling per tree
    reg_alpha         = 0.1,    # L1 regularization
    reg_lambda        = 0.2,    # L2 regularization
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1
)

print('⏳ Training LightGBM...')
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

y_pred = np.maximum(model.predict(X_test), 0)  # no negative predictions
print(f'\n✅ Training complete — used {model.n_estimators_} trees (early stopped)')

## Evaluate Model

In [ ]:
# ── Overall Metrics ───────────────────────────────────────────────────────────
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
r2   = r2_score(y_test, y_pred)

print('='*52)
print('  📊 FEATURE 1 — MODEL EVALUATION RESULTS')
print('='*52)
print(f'  MAE  (Mean Absolute Error)    : {mae:.3f} units/day')
print(f'  RMSE (Root Mean Sq Error)     : {rmse:.3f} units/day')
print(f'  MAPE (Mean Abs % Error)       : {mape:.2f}%')
print(f'  R²   (Coefficient of Det.)    : {r2:.4f}')
print('='*52)
print()
print('  How to interpret these for your report:')
print(f'  → On average the model is off by {mae:.1f} units/day')
print(f'  → That is {mape:.1f}% relative to actual demand')
print(f'  → R² of {r2:.2f} means the model explains {r2*100:.0f}% of demand variation')

# ── Per-Product MAPE ─────────────────────────────────────────────────────────
test_results = test_df[['date','pharmacy_id','sku_id','product_name','quantity_sold']].copy()
test_results['predicted'] = y_pred
test_results['abs_error'] = abs(test_results['quantity_sold'] - test_results['predicted'])
test_results['pct_error'] = test_results['abs_error'] / (test_results['quantity_sold'] + 1e-6) * 100

per_product = (
    test_results
    .groupby('product_name')
    .agg(
        mape=('pct_error', 'mean'),
        mae=('abs_error',  'mean'),
        n_rows=('quantity_sold', 'count')
    )
    .round(2)
    .sort_values('mape')
)

print('\n  Per-Product MAPE (lower = better):')
print(per_product[['mape','mae']].to_string())

In [ ]:
# ── Evaluation Charts ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Feature 1 — LightGBM Demand Forecasting: Evaluation', fontsize=14, fontweight='bold')

# 1. Actual vs Predicted scatter
sample_idx = np.random.choice(len(y_test), min(1000, len(y_test)), replace=False)
axes[0,0].scatter(
    y_test.values[sample_idx], y_pred[sample_idx],
    alpha=0.3, s=15, color='#7B1FA2'
)
max_val = max(y_test.max(), y_pred.max())
axes[0,0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0,0].set_title(f'Actual vs Predicted  (R²={r2:.3f})')
axes[0,0].set_xlabel('Actual Quantity Sold')
axes[0,0].set_ylabel('Predicted Quantity')
axes[0,0].legend()

# 2. Feature importance (top 15)
importance = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True).tail(15)
axes[0,1].barh(importance['feature'], importance['importance'], color='#AB47BC')
axes[0,1].set_title('Top 15 Feature Importances')
axes[0,1].set_xlabel('Importance Score')

# 3. Residual distribution
residuals = y_test.values - y_pred
axes[1,0].hist(residuals, bins=60, color='#7B1FA2', alpha=0.75, edgecolor='white')
axes[1,0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero error')
axes[1,0].set_title('Prediction Error Distribution (Residuals)')
axes[1,0].set_xlabel('Actual − Predicted')
axes[1,0].set_ylabel('Count')
axes[1,0].legend()

# 4. Per-product MAPE bar chart
per_product_sorted = per_product.sort_values('mape', ascending=True)
short_names = [n.replace(' with ', ' w/ ')[:22] for n in per_product_sorted.index]
bars = axes[1,1].barh(short_names, per_product_sorted['mape'], color='#CE93D8')
axes[1,1].axvline(mape, color='red', linestyle='--', linewidth=2, label=f'Overall MAPE ({mape:.1f}%)')
axes[1,1].set_title('MAPE by Product')
axes[1,1].set_xlabel('MAPE (%)')
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('/content/ML/models/evaluation_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation charts saved')

## Baseline Comparison (Holt-Winters vs LightGBM)

In [ ]:
# ── We pick one pharmacy + one product to show the comparison visually ────────
# This goes directly into your project report as proof LightGBM > naive baseline

DEMO_PHARMACY = 'PH_01'
DEMO_SKU      = 'SKU_004'  # Day Pads - Pack of 6 (highest weekly demand)

ts_data = (
    sales_df[
        (sales_df['pharmacy_id'] == DEMO_PHARMACY) &
        (sales_df['sku_id']      == DEMO_SKU)
    ]
    .set_index('date')['quantity_sold']
    .resample('W').sum()  # weekly aggregation — smoother for HW
)

split_idx  = int(len(ts_data) * 0.8)
hw_train   = ts_data.iloc[:split_idx]
hw_test    = ts_data.iloc[split_idx:]

# Train Holt-Winters
hw_model = ExponentialSmoothing(
    hw_train,
    trend='add',
    seasonal='add',
    seasonal_periods=12  # ~12 weeks quarterly
).fit(optimized=True)

hw_forecast = np.maximum(hw_model.forecast(len(hw_test)), 0)
hw_mape = mean_absolute_percentage_error(hw_test, hw_forecast) * 100

# LightGBM on same scope
lgb_subset = test_results[
    (test_results['pharmacy_id'] == DEMO_PHARMACY) &
    (test_results['sku_id']      == DEMO_SKU)
]
lgb_mape_subset = lgb_subset['pct_error'].mean() if len(lgb_subset) > 0 else mape

# ── Print comparison ─────────────────────────────────────────────────────────
print('='*55)
print('  MODEL COMPARISON: Day Pads – Pack of 6 @ PH_01')
print('='*55)
print(f'  Holt-Winters MAPE : {hw_mape:.2f}%  (simple baseline)')
print(f'  LightGBM MAPE     : {lgb_mape_subset:.2f}%  (our model)')
print(f'  Improvement       : {hw_mape - lgb_mape_subset:.2f}% better')
print(f'  Improvement ratio : {hw_mape/max(lgb_mape_subset,0.1):.2f}x')
print('='*55)

# ── Visual comparison ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hw_train.index, hw_train.values, color='#9E9E9E', linewidth=1.5, label='Train (actual)')
ax.plot(hw_test.index,  hw_test.values,  color='#4A148C', linewidth=2.5, label='Actual (test)')
ax.plot(
    hw_test.index, hw_forecast.values,
    color='#FF6F00', linewidth=2, linestyle='--',
    label=f'Holt-Winters (MAPE={hw_mape:.1f}%)'
)
ax.axvspan(hw_test.index[0], hw_test.index[-1], alpha=0.05, color='purple', label='Test window')
ax.set_title(
    'Baseline Comparison — Day Pads (Apollo Pharmacy, Delhi)\n'
    f'Holt-Winters MAPE={hw_mape:.1f}%  vs  LightGBM MAPE={lgb_mape_subset:.1f}%',
    fontweight='bold'
)
ax.set_ylabel('Weekly Units Sold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/ML/models/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Baseline chart saved — use this in your report!')

## Predict Next 7 Days Demand

In [ ]:
def predict_next_7_days(pharmacy_id, sku_id, feat_df, model, FEATURE_COLS):
    """
    For a given pharmacy + product:
    Takes their most recent sales row (latest features)
    Predicts daily demand and sums to 7-day forecast.

    Returns dict with all forecast details.
    """
    subset = feat_df[
        (feat_df['pharmacy_id'] == pharmacy_id) &
        (feat_df['sku_id']      == sku_id)
    ].sort_values('date')

    if len(subset) == 0:
        return None

    latest_row = subset.iloc[-1]
    features   = latest_row[FEATURE_COLS].values.reshape(1, -1)
    daily_pred = float(np.maximum(model.predict(features)[0], 0))

    # 7-day forecast = daily × 7
    # Confidence interval = ±1.5 × rolling std (simple heuristic)
    roll_std   = latest_row.get('roll_7_std', daily_pred * 0.2)
    if pd.isna(roll_std):
        roll_std = daily_pred * 0.2

    forecast_7d   = daily_pred * 7
    ci_lower      = max(0, (daily_pred - 1.5 * roll_std) * 7)
    ci_upper      = (daily_pred + 1.5 * roll_std) * 7

    # Confidence level based on how stable demand has been
    cv = roll_std / (daily_pred + 1e-6)
    if cv < 0.3:
        confidence = 'High'
    elif cv < 0.6:
        confidence = 'Medium'
    else:
        confidence = 'Low'

    return {
        'pharmacy_id':   pharmacy_id,
        'sku_id':        sku_id,
        'product_name':  latest_row['product_name'],
        'pharmacy_name': latest_row['pharmacy_name'],
        'city':          latest_row['city'],
        'daily_pred':    round(daily_pred, 2),
        'forecast_7d':   round(forecast_7d),
        'ci_lower_7d':   round(ci_lower),
        'ci_upper_7d':   round(ci_upper),
        'confidence':    confidence,
        'unit_price':    int(latest_row['unit_price']),
    }


# ── Run predictions for ALL pharmacies × ALL products ─────────────────────────
print('⏳ Running predictions for all 15 pharmacies × 7 products...')

all_forecasts = []
for ph_id in sales_df['pharmacy_id'].unique():
    for sku_id in sales_df['sku_id'].unique():
        result = predict_next_7_days(ph_id, sku_id, feat_df, model, FEATURE_COLS)
        if result:
            all_forecasts.append(result)

forecasts_df = pd.DataFrame(all_forecasts)
print(f'✅ Generated {len(forecasts_df)} forecasts')
print()
print('Sample forecasts:')
forecasts_df[['pharmacy_name','product_name','forecast_7d','ci_lower_7d','ci_upper_7d','confidence']].head(7)

## Generate Reorder Suggestions

In [ ]:
def generate_reorder_suggestions(forecasts_df, inventory_df):
    """
    Joins 7-day demand forecast with current inventory.
    If predicted demand > current stock → generate reorder suggestion.

    Reorder formula:
        reorder_qty = (forecast_7d × safety_factor) − current_stock
        safety_factor = 1.2  (20% buffer for uncertainty)
    """
    SAFETY_FACTOR = 1.2

    # Merge forecasts with current inventory
    merged = forecasts_df.merge(
        inventory_df[['pharmacy_id','sku_id','current_stock','reorder_level',
                      'days_of_cover','is_low_stock','is_out_of_stock','distributor_id']],
        on=['pharmacy_id','sku_id'],
        how='left'
    )

    # Calculate reorder quantity
    merged['needed_qty']   = (merged['forecast_7d'] * SAFETY_FACTOR).apply(np.ceil)
    merged['reorder_qty']  = (merged['needed_qty'] - merged['current_stock']).clip(lower=0).astype(int)
    merged['should_order'] = merged['reorder_qty'] > 0

    # Days until stockout
    merged['days_until_stockout'] = (
        merged['current_stock'] / merged['daily_pred'].clip(lower=0.01)
    ).round(1)

    # Urgency level
    def urgency(row):
        if row['is_out_of_stock']:  return 'CRITICAL — Out of Stock'
        if row['days_until_stockout'] <= 2:  return 'URGENT — < 2 days'
        if row['days_until_stockout'] <= 5:  return 'HIGH — < 5 days'
        if row['should_order']:              return 'MEDIUM — Order Soon'
        return 'OK'

    merged['urgency']       = merged.apply(urgency, axis=1)
    merged['order_value']   = merged['reorder_qty'] * merged['unit_price']

    return merged


suggestions = generate_reorder_suggestions(forecasts_df, inventory_df)

# ── Summary stats ─────────────────────────────────────────────────────────────
needs_order = suggestions[suggestions['should_order']]
critical    = suggestions[suggestions['urgency'].str.startswith('CRITICAL')]
urgent      = suggestions[suggestions['urgency'].str.startswith('URGENT')]

print('='*55)
print('  📦 REORDER SUGGESTIONS SUMMARY')
print('='*55)
print(f'  Total pharmacy-product pairs : {len(suggestions)}')
print(f'  Need reorder                 : {len(needs_order)}')
print(f'  CRITICAL (out of stock)      : {len(critical)}')
print(f'  URGENT (< 2 days left)       : {len(urgent)}')
print(f'  Total reorder value          : ₹{needs_order["order_value"].sum():,.0f}')
print('='*55)
print()

# Show urgent ones
urgent_view = needs_order.sort_values('days_until_stockout').head(10)
print('Most Urgent Reorders:')
urgent_view[['pharmacy_name','product_name','current_stock','forecast_7d',
             'reorder_qty','days_until_stockout','urgency','order_value']].to_string(index=False)

In [ ]:
# ── Reorder Dashboard Chart ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('📦 Reorder Suggestion Dashboard — All Pharmacies', fontsize=14, fontweight='bold')

# 1. Urgency breakdown
urgency_counts = suggestions['urgency'].value_counts()
urgency_colors = {
    'OK': '#43A047',
    'MEDIUM — Order Soon': '#FB8C00',
    'HIGH — < 5 days': '#F4511E',
    'URGENT — < 2 days': '#B71C1C',
    'CRITICAL — Out of Stock': '#4A148C'
}
colors = [urgency_colors.get(u, 'gray') for u in urgency_counts.index]
axes[0].barh(urgency_counts.index, urgency_counts.values, color=colors)
axes[0].set_title('Reorder Urgency Breakdown')
axes[0].set_xlabel('Number of SKUs')

# 2. Top pharmacies by total reorder value
ph_reorder = (
    needs_order
    .groupby('pharmacy_name')['order_value']
    .sum()
    .sort_values(ascending=True)
    .tail(10)
)
short_ph = [n.split(' - ')[0][:20] for n in ph_reorder.index]
axes[1].barh(short_ph, ph_reorder.values / 1000, color='#7B1FA2', alpha=0.8)
axes[1].set_title('Top Pharmacies by Reorder Value')
axes[1].set_xlabel('Reorder Value (₹ Thousands)')

# 3. Per-product reorder quantity total
prod_reorder = (
    needs_order
    .groupby('product_name')['reorder_qty']
    .sum()
    .sort_values(ascending=True)
)
short_prod = [p.replace(' with ', ' w/ ')[:22] for p in prod_reorder.index]
axes[2].barh(short_prod, prod_reorder.values, color='#AB47BC', alpha=0.8)
axes[2].set_title('Total Reorder Qty by Product')
axes[2].set_xlabel('Total Units to Order')

plt.tight_layout()
plt.savefig('/content/ML/models/reorder_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# Save CSV for FastAPI to use
suggestions.to_csv('/content/ML/models/reorder_suggestions.csv', index=False)
print('✅ reorder_suggestions.csv saved')

## Gemini Explainability

In [ ]:
import google.generativeai as genai

# Get your FREE key from: https://makersuite.google.com/app/apikey
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'  # ← paste your key here

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

print('✅ Gemini client ready')

In [ ]:
def explain_reorder(row):
    """
    Takes one reorder suggestion row.
    Sends the numbers to Gemini.
    Returns a plain-language explanation for the pharmacy dashboard.
    """
    prompt = f"""You are a supply assistant for an Indian pharmacy B2B platform called Swasthya.
Write a SHORT 2-sentence reorder recommendation for the pharmacist.
Use simple clear language. No bullet points. No markdown.
Mention the product name and urgency naturally.

Context:
- Pharmacy     : {row['pharmacy_name']}, {row['city']}
- Product      : {row['product_name']}
- Current stock: {row['current_stock']} units
- Predicted demand next 7 days: {row['forecast_7d']} units (range: {row['ci_lower_7d']}–{row['ci_upper_7d']})
- Days until stockout: {row['days_until_stockout']} days
- Suggested reorder quantity: {row['reorder_qty']} units
- Urgency: {row['urgency']}
- Forecast confidence: {row['confidence']}

Speak directly to the pharmacist. Be concise and actionable."""

    try:
        response = gemini.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f'Reorder {row["reorder_qty"]} units of {row["product_name"]} — stock will last {row["days_until_stockout"]} days.'


# ── Demo: Show 5 explanations for the most urgent reorders ───────────────────
demo_rows = needs_order.sort_values('days_until_stockout').head(5)

print('💬 GEMINI REORDER EXPLANATIONS (shown on pharmacy dashboard)\n')
print('─'*60)

for _, row in demo_rows.iterrows():
    explanation = explain_reorder(row)
    print(f'\n🏥 {row["pharmacy_name"]}')
    print(f'📦 {row["product_name"]} | Stock: {row["current_stock"]} | Reorder: {row["reorder_qty"]} units')
    print(f'⚠️  {row["urgency"]}')
    print(f'🤖 AI: {explanation}')
    print('─'*60)

In [ ]:
# ── Full inference function (what FastAPI will call) ──────────────────────────
def get_reorder_suggestion(pharmacy_id: str, sku_id: str,
                           current_stock: int,
                           feat_df, model, FEATURE_COLS,
                           inventory_df, use_gemini: bool = True):
    """
    Single function your FastAPI /forecast endpoint calls.

    Input  : pharmacy_id, sku_id, current_stock
    Output : full reorder suggestion dict with optional Gemini explanation
    """
    # Step 1: Get forecast
    forecast = predict_next_7_days(pharmacy_id, sku_id, feat_df, model, FEATURE_COLS)
    if forecast is None:
        return {'error': 'No data found for this pharmacy+SKU combination'}

    # Step 2: Get current stock from inventory (or use passed value)
    inv_row = inventory_df[
        (inventory_df['pharmacy_id'] == pharmacy_id) &
        (inventory_df['sku_id']      == sku_id)
    ]
    if len(inv_row) > 0:
        current_stock = int(inv_row.iloc[0]['current_stock'])
        reorder_level = int(inv_row.iloc[0]['reorder_level'])
    else:
        reorder_level = int(forecast['forecast_7d'] * 0.5)

    # Step 3: Calculate reorder qty
    SAFETY_FACTOR  = 1.2
    needed_qty     = int(np.ceil(forecast['forecast_7d'] * SAFETY_FACTOR))
    reorder_qty    = max(0, needed_qty - current_stock)
    should_order   = reorder_qty > 0
    days_until_out = round(current_stock / max(forecast['daily_pred'], 0.01), 1)

    if days_until_out == 0:
        urgency = 'CRITICAL — Out of Stock'
    elif days_until_out <= 2:
        urgency = 'URGENT — < 2 days'
    elif days_until_out <= 5:
        urgency = 'HIGH — < 5 days'
    elif should_order:
        urgency = 'MEDIUM — Order Soon'
    else:
        urgency = 'OK'

    result = {
        **forecast,
        'current_stock':      current_stock,
        'reorder_level':      reorder_level,
        'reorder_qty':        reorder_qty,
        'should_order':       should_order,
        'days_until_stockout': days_until_out,
        'urgency':            urgency,
        'order_value':        reorder_qty * forecast['unit_price'],
        'explanation':        None
    }

    # Step 4: Gemini explanation (only if reorder needed)
    if use_gemini and should_order:
        result['explanation'] = explain_reorder(pd.Series(result))

    return result


# ── Test the full function ────────────────────────────────────────────────────
print('🔬 Testing full inference function...\n')
test_result = get_reorder_suggestion(
    pharmacy_id   = 'PH_01',
    sku_id        = 'SKU_004',
    current_stock = 15,
    feat_df       = feat_df,
    model         = model,
    FEATURE_COLS  = FEATURE_COLS,
    inventory_df  = inventory_df,
    use_gemini    = True
)

for key, value in test_result.items():
    print(f'  {key:25s}: {value}')

## Save All Model Files

In [ ]:
# ── Save trained model ────────────────────────────────────────────────────────
joblib.dump(model, '/content/ML/models/demand_forecaster.pkl')

# ── Save feature list + metadata ──────────────────────────────────────────────
metadata = {
    'project':      'Swasthya — Intelligent Pharmacy Network',
    'feature':      'Feature 1 — Pharmacy Demand Forecasting',
    'model_type':   'LightGBMRegressor',
    'feature_cols': FEATURE_COLS,
    'target_col':   TARGET_COL,
    'n_features':   len(FEATURE_COLS),
    'n_trees_used': int(model.n_estimators_),
    'train_rows':   len(X_train),
    'test_rows':    len(X_test),
    'train_period': f'{train_df["date"].min().date()} to {train_df["date"].max().date()}',
    'test_period':  f'{test_df["date"].min().date()} to {test_df["date"].max().date()}',
    'metrics': {
        'MAE':  round(mae, 4),
        'RMSE': round(rmse, 4),
        'MAPE': round(mape, 4),
        'R2':   round(r2, 4),
        'baseline_holt_winters_mape': round(hw_mape, 4),
        'improvement_over_baseline':  round(hw_mape - mape, 4)
    },
    'products': [
        {'sku_id': p['sku_id'], 'name': p['name'], 'price': p['price']}
        for p in products_df.to_dict('records')
    ],
    'n_pharmacies': int(sales_df['pharmacy_id'].nunique()),
    'safety_factor': 1.2,
    'forecast_horizon_days': 7,
}

with open('/content/ML/models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# ── List saved files ──────────────────────────────────────────────────────────
print('✅ All files saved to /content/ML/models/')
print()
for fname in os.listdir('/content/ML/models/'):
    fpath = f'/content/ML/models/{fname}'
    size  = os.path.getsize(fpath)
    print(f'  {fname:40s}  {size/1024:.1f} KB')

# ── Download zip ──────────────────────────────────────────────────────────────
import zipfile
from google.colab import files

with zipfile.ZipFile('/content/feature1_models.zip', 'w') as zf:
    for fname in os.listdir('/content/ML/models/'):
        zf.write(f'/content/ML/models/{fname}', fname)

print('\n📦 Downloading feature1_models.zip...')
files.download('/content/feature1_models.zip')

## FastAPI-Ready Inference Function

In [ ]:
# ── This is what your ML/api/main.py will look like ──────────────────────────
# Copy this block into your FastAPI file

FASTAPI_CODE = '''
# ML/api/main.py
# Feature 1: Pharmacy Demand Forecasting Endpoint

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import json
import numpy as np
import pandas as pd
import google.generativeai as genai
import os

app = FastAPI(title="Swasthya ML API — Feature 1")

# Load model and metadata once at startup
model    = joblib.load("models/demand_forecaster.pkl")
metadata = json.load(open("models/model_metadata.json"))
FEATURES = metadata["feature_cols"]

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
gemini = genai.GenerativeModel("gemini-1.5-flash")


class ForecastRequest(BaseModel):
    pharmacy_id:    str
    sku_id:         str
    current_stock:  int
    # Feature values — your Expo app reads these from local SQLite
    lag_1:          float
    lag_7:          float
    lag_14:         float
    lag_28:         float
    lag_91:         float
    roll_7_mean:    float
    roll_7_std:     float
    roll_14_mean:   float
    roll_30_mean:   float
    roll_60_mean:   float
    roll_90_mean:   float
    trend_7_vs_30:  float
    trend_14_vs_60: float
    trend_30_vs_90: float
    sin_dow:        float
    cos_dow:        float
    sin_doy:        float
    cos_doy:        float
    sin_month:      float
    cos_month:      float
    month:          int
    quarter:        int
    city_tier:      int
    is_weekend:     int
    is_mhd_window:  int
    is_ngo_season:  int
    is_monsoon:     int
    sku_code:       int
    ph_code:        int
    city_code:      int
    cat_code:       int
    unit_price:     int
    product_name:   str
    pharmacy_name:  str
    city:           str


@app.post("/forecast")
async def forecast(req: ForecastRequest):
    features = np.array([[
        req.lag_1, req.lag_7, req.lag_14, req.lag_28, req.lag_91,
        req.roll_7_mean, req.roll_7_std, req.roll_14_mean,
        req.roll_30_mean, req.roll_60_mean, req.roll_90_mean,
        req.trend_7_vs_30, req.trend_14_vs_60, req.trend_30_vs_90,
        req.sin_dow, req.cos_dow, req.sin_doy, req.cos_doy,
        req.sin_month, req.cos_month,
        req.month, req.quarter, req.city_tier,
        req.is_weekend, req.is_mhd_window, req.is_ngo_season, req.is_monsoon,
        req.sku_code, req.ph_code, req.city_code, req.cat_code
    ]])

    daily_pred   = float(np.maximum(model.predict(features)[0], 0))
    forecast_7d  = round(daily_pred * 7)
    reorder_qty  = max(0, round(forecast_7d * 1.2) - req.current_stock)
    should_order = reorder_qty > 0
    days_left    = round(req.current_stock / max(daily_pred, 0.01), 1)

    if days_left == 0:      urgency = "CRITICAL"
    elif days_left <= 2:    urgency = "URGENT"
    elif days_left <= 5:    urgency = "HIGH"
    elif should_order:      urgency = "MEDIUM"
    else:                   urgency = "OK"

    explanation = None
    if should_order:
        prompt = f"""
        Swasthya pharmacy assistant. Write 2-sentence reorder recommendation.
        Pharmacy: {req.pharmacy_name}, {req.city}
        Product: {req.product_name}
        Stock: {req.current_stock} units | Predicted 7d: {forecast_7d} units
        Days left: {days_left} | Reorder: {reorder_qty} units | Urgency: {urgency}
        """
        explanation = gemini.generate_content(prompt).text

    return {
        "pharmacy_id":        req.pharmacy_id,
        "sku_id":             req.sku_id,
        "daily_pred":         round(daily_pred, 2),
        "forecast_7d":        forecast_7d,
        "current_stock":      req.current_stock,
        "reorder_qty":        reorder_qty,
        "should_order":       should_order,
        "days_until_stockout": days_left,
        "urgency":            urgency,
        "order_value":        reorder_qty * req.unit_price,
        "explanation":        explanation,
    }


@app.get("/health")
def health():
    return {"status": "ok", "model": "demand_forecaster", "mape": metadata["metrics"]["MAPE"]}
'''

print(FASTAPI_CODE)
print('\n✅ Copy the above into ML/api/main.py in your repo')

---
## Feature 1 : Demand Forecasting and Reorder Suggestions

### Files saved:
```
ML/
└── models/
    ├── demand_forecaster.pkl       ← LightGBM trained model
    ├── model_metadata.json         ← metrics + feature list
    ├── reorder_suggestions.csv     ← all pharmacy reorder suggestions
    ├── eda_charts.png              ← for report
    ├── evaluation_charts.png       ← for report
    ├── baseline_comparison.png     ← for report (LightGBM vs Holt-Winters)
    └── reorder_dashboard.png       ← for report
```